# Day 24 — CNNs, Visualized

A conv net detects local spatial patterns. We'll classify 'bright row' vs 'bright column'
8x8 images — trivial for convolution, hard for a flat MLP.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

def make_images(n=120):
    imgs, labels = [], []
    for i in range(n):
        img = torch.rand(1, 8, 8) * 0.1
        if i % 2 == 0: img[0, torch.randint(0, 8, (1,)).item(), :] += 1.0; labels.append(0)
        else: img[0, :, torch.randint(0, 8, (1,)).item()] += 1.0; labels.append(1)
        imgs.append(img)
    return torch.stack(imgs), torch.tensor(labels)
X, y = make_images()
print('data:', X.shape, ' labels:', y[:6].tolist(), '(0=row, 1=column)')

## A few samples

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for ax, idx in zip(axes, [0, 1, 2, 3]):
    ax.imshow(X[idx, 0], cmap='gray'); ax.set_title(f'class {y[idx].item()}'); ax.axis('off')
plt.show()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, 3, padding=1); self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(8 * 4 * 4, 2)
    def forward(self, x):
        return self.fc(self.pool(torch.relu(self.conv(x))).flatten(1))

model = CNN(); opt = torch.optim.Adam(model.parameters(), lr=0.01); crit = nn.CrossEntropyLoss()
for _ in range(80):
    opt.zero_grad(); crit(model(X), y).backward(); opt.step()
print('accuracy:', (model(X).argmax(1) == y).float().mean().item())

## Takeaways

- Convolution shares filter weights across the image -> detects a pattern anywhere.
- Watch the flattened size (8 channels x 4 x 4 = 128) into the final Linear — the classic bug.

**Now build** SmallCNN / build_cnn / train in `homework.py`, then `pytest -q`.